# Study 889 — Broad Dollar-Hedge Overlay — the teardown

The carry decomposition, the HAC *t* on the same-basket pair, the β ≈ 1 hedge regression, the 2022 era split, the excess-of-cash Sharpe race, the costed overlay, the UUP collateral-yield trap, and the planted-carry synthetic control.

In [1]:
R = {'fp': '7afa89fb9f2f', 'asof': '2026-06-30', 'h_n': 148, 'h_start': '2014-03', 'h_carry': 1.68, 'h_t': 4.74, 'h_obs': 1.35, 'h_beta': 0.93, 'h_tbeta': 34.8, 'h_alpha': 1.79, 'h_talpha': 5.38, 'h_r2': 0.91, 'h_ci_lo': 0.9, 'h_ci_hi': 2.45, 'h_frac0': 0.0, 'h_pre_carry': 1.15, 'h_pre_t': 2.89, 'h_pre_n': 94, 'h_pre_obs': 0.89, 'h_post_carry': 2.61, 'h_post_t': 4.88, 'h_post_n': 54, 'h_post_obs': 2.14, 'h_sh': 0.75, 'h_sh_lo': 0.2, 'h_sh_hi': 1.35, 'u_sh': 0.4, 'u_sh_lo': -0.11, 'u_sh_hi': 0.95, 'adv': 0.35, 'h_dd': -20.7, 'u_dd': -27.6, 'd_n': 180, 'd_carry': 1.37, 'd_t': 2.56, 'd_obs': 1.09, 'd_beta': 0.9, 'd_r2': 0.69, 'd_ci_lo': -0.02, 'd_ci_hi': 2.84, 'ov_switches': 1, 'ov_share': 0.93, 'ov_sh': 0.67, 'ov_hedged': 0.75, 'ov_unhedged': 0.4, 'ov_adv_u': 0.27, 'ov_adv_h': -0.08, 'ov_cost': 0.005, 'sp_net': 2.53, 'sp_t': 1.41, 'sp_charge': 0.62, 'null_mean': 0.03, 'null_sd': 0.89, 'null_fire': 0, 'null_seeds': 20, 'planted_carry': 2.56, 'planted_t': 7.08, 'planted_beta': 1.0}

## The identity — `carry_hat = (hedged − unhedged) + fx_foreign`

`fx_foreign` is the **spot** USD return of an EAFE-weighted EUR/JPY/GBP/CHF basket. The hedge regression is `diff = α + β·(−fx_foreign)`: β ≈ 1 = a full short of the basket.

In [2]:
print(f"HEFA/EFA (same basket, n={R['h_n']}):")
print(f"  carry_hat {R['h_carry']:+.2f}%/yr  HAC t = {R['h_t']:+.2f}   "
      f"observed rate diff {R['h_obs']:+.2f}%/yr")
print(f"  hedge reg: beta = {R['h_beta']:.2f} (t {R['h_tbeta']:+.1f}), "
      f"alpha = {R['h_alpha']:+.2f}%/yr (t {R['h_talpha']:+.2f}), R2 = {R['h_r2']:.2f}")
print(f"  carry bootstrap 95% CI [{R['h_ci_lo']:+.2f}, {R['h_ci_hi']:+.2f}]  "
      f"frac<=0 {R['h_frac0']:.3f}")
print(f"DBEF/EFA (provider diff, n={R['d_n']}): carry {R['d_carry']:+.2f}%/yr "
      f"HAC t = {R['d_t']:+.2f}  beta {R['d_beta']:.2f}  R2 {R['d_r2']:.2f}  "
      f"CI [{R['d_ci_lo']:+.2f}, {R['d_ci_hi']:+.2f}]")

HEFA/EFA (same basket, n=148):
  carry_hat +1.68%/yr  HAC t = +4.74   observed rate diff +1.35%/yr
  hedge reg: beta = 0.93 (t +34.8), alpha = +1.79%/yr (t +5.38), R2 = 0.91
  carry bootstrap 95% CI [+0.90, +2.45]  frac<=0 0.000
DBEF/EFA (provider diff, n=180): carry +1.37%/yr HAC t = +2.56  beta 0.90  R2 0.69  CI [-0.02, +2.84]


## Era split (cut 2022-01-01) — the carry grows with the differential

In [3]:
print(f"HEFA/EFA pre-2022 (n={R['h_pre_n']}): carry {R['h_pre_carry']:+.2f}%/yr "
      f"HAC t = {R['h_pre_t']:+.2f}  (obs diff {R['h_pre_obs']:+.2f})")
print(f"HEFA/EFA 2022+   (n={R['h_post_n']}): carry {R['h_post_carry']:+.2f}%/yr "
      f"HAC t = {R['h_post_t']:+.2f}  (obs diff {R['h_post_obs']:+.2f})")
print('  -> clears t>=2 in BOTH eras and grows as the gap widens: the mechanical signature')

HEFA/EFA pre-2022 (n=94): carry +1.15%/yr HAC t = +2.89  (obs diff +0.89)
HEFA/EFA 2022+   (n=54): carry +2.61%/yr HAC t = +4.88  (obs diff +2.14)
  -> clears t>=2 in BOTH eras and grows as the gap widens: the mechanical signature


## The excess-of-cash Sharpe race (both legs minus BIL)

The hedged sleeve out-Sharped the unhedged one and cut the drawdown — but the CIs **overlap**, so the *advantage* rests on one realised dollar regime, not on the carry.

In [4]:
print(f"hedged   ex-cash Sharpe {R['h_sh']:.2f}  95% CI [{R['h_sh_lo']:.2f}, {R['h_sh_hi']:.2f}]")
print(f"unhedged ex-cash Sharpe {R['u_sh']:.2f}  95% CI [{R['u_sh_lo']:.2f}, {R['u_sh_hi']:.2f}]")
print(f"advantage {R['adv']:+.2f} (CIs overlap)   max DD hedged {R['h_dd']:.1f}% vs unhedged {R['u_dd']:.1f}%")

hedged   ex-cash Sharpe 0.75  95% CI [0.20, 1.35]
unhedged ex-cash Sharpe 0.40  95% CI [-0.11, 0.95]
advantage +0.35 (CIs overlap)   max DD hedged -20.7% vs unhedged -27.6%


## The overlay and the isolation spread — why it is *held*, not *timed*

In [5]:
print(f"overlay: {R['ov_switches']} switch, share hedged {R['ov_share']:.2f}, "
      f"Sharpe {R['ov_sh']:.2f} vs always-hedged {R['ov_hedged']:.2f} "
      f"(adv vs hedged {R['ov_adv_h']:+.2f}; cost drag {R['ov_cost']:.3f}%/yr)")
print(f"isolation spread (long hedged/short unhedged = long the dollar): "
      f"net {R['sp_net']:+.2f}%/yr  HAC t = {R['sp_t']:+.2f}  (fx vol swamps the carry)")

overlay: 1 switch, share hedged 0.93, Sharpe 0.67 vs always-hedged 0.75 (adv vs hedged -0.08; cost drag 0.005%/yr)
isolation spread (long hedged/short unhedged = long the dollar): net +2.53%/yr  HAC t = +1.41  (fx vol swamps the carry)


## The UUP trap — why we use spot, not the tradeable dollar ETF

`UUP`'s total return also earns the US-bill collateral yield, so a naive `carry_hat = diff − UUP` subtracts that yield and **cancels most of the carry** — it prints *negative*, worst in the high-rate 2022+ era. The live cell shows it on the synthetic world's clean fx leg vs a collateral-contaminated dollar.

In [6]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from dollar_hedge import data, strategy as st
w = data.synthetic_world(n_months=180, carry_annual=0.03, seed=889)
pf = st.pair_frame(w, 'HEFA', 'EFA')
clean = st.nw_mean_t((pf['diff'] + pf['fx_foreign']).values)[0]*12*100
# a UUP-like dollar that ALSO earns ~4%/yr collateral: subtracting it eats the carry
uup_like = -pf['fx_foreign'] + 0.04/12
contaminated = st.nw_mean_t((pf['diff'] - uup_like).values)[0]*12*100
print(f'carry via SPOT fx basket   : {clean:+.2f}%/yr  (recovers the planted +3%)')
print(f'carry via UUP-like (w/ 4pct yield): {contaminated:+.2f}%/yr  (collateral yield cancels it)')

carry via SPOT fx basket   : +2.56%/yr  (recovers the planted +3%)
carry via UUP-like (w/ 4pct yield): -1.44%/yr  (collateral yield cancels it)


## Synthetic positive control — the machinery is unbiased

Live: the estimator must NOT fire on the null and must recover a planted carry with β ≈ 1.

In [7]:
null_t = np.array([st.synthetic_detect(
    data.synthetic_world(n_months=180, carry_annual=0.0, seed=889+s))['t_carry'] for s in range(8)])
print(f"null (carry=0), 8 seeds: HAC t mean {null_t.mean():+.2f} (sd {null_t.std(ddof=1):.2f}), "
      f"|t|>=2 in {(abs(null_t)>=2).sum()}/8")
planted = st.synthetic_detect(data.synthetic_world(n_months=180, carry_annual=0.03, seed=889))
print(f"planted (+3%/yr): recovered {planted['carry_ann_pct']:+.2f}%/yr "
      f"(HAC t {planted['t_carry']:+.2f}), hedge beta {planted['beta']:.2f}")
print(f"(frozen full 20-seed run: null t mean {R['null_mean']:+.2f}, fires {R['null_fire']}/{R['null_seeds']})")

null (carry=0), 8 seeds: HAC t mean +0.11 (sd 0.75), |t|>=2 in 0/8
planted (+3%/yr): recovered +2.56%/yr (HAC t +7.08), hedge beta 1.00
(frozen full 20-seed run: null t mean +0.03, fires 0/20)


## Verdict

- **Signal — Real.** The 613 currency-hedge carry identity **generalises to broad developed international.** On the clean same-basket HEFA/EFA pair the hedge is a near-full short of the foreign basket (β = 0.93, R² = 0.91) and pockets **+1.68 %/yr at HAC *t* = +4.74** — on the observable +1.35 %/yr differential, bootstrap CI [+0.90, +2.45] clear of zero, *t* ≥ 2 in both eras (+2.89 / +4.88), growing with the gap. DBEF/EFA corroborates (*t* = +2.56). The synthetic control recovers a *planted* carry (β = 1.00, fires 0/20 nulls).
- **Tradability — Fragile.** The carry is real and cheap to *hold* (~2 bp wrapper gap), but **not bankable as an overlay**: the 'hedge when the US out-yields' switch adds nothing over just always-hedging (0.67 vs 0.75 Sharpe, 1 switch in 12 yr); the excess-Sharpe advantage's CI overlaps and rests on one dollar regime; isolating the pure carry is a dollar-long spread that nets +2.53 %/yr at only *t* = +1.41. Real but thin & un-timeable.